<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/05_GES_Aware_Genomic_RAG_Cell_7B4_Configuration_Freeze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GES-RAG Experiment 2 — Cell 7B4
## Embedding, Retrieval, Reranking, Prompt, LLM, Runtime, and Determinism Configuration Freeze

**Authorized scope:** configuration freeze only.

This notebook:

- cryptographically reverifies the frozen Cell 7B3 package;
- reads only file/checksum metadata, Parquet schema metadata, CSV row counts, and the Cell 7B3 manifest authorization;
- freezes the exact embedding model and revision, normalization, similarity search, top-20 candidate-pool implementation, top-5 context policy, quality reranking, blinded aliases, prompt templates, strict response schema, fixed LLM snapshot, generation settings, runtime package targets, and deterministic controls;
- writes checksum-protected Cell 7B4 configuration artifacts.

**This notebook does not generate embeddings, run retrieval, apply reranking, materialize prompts, call an LLM, inspect answer-key outcomes, tune weights, replace questions, or evaluate RAG performance.**

The next execution stage remains unauthorized until a separate fail-closed authorization cell is created.

In [1]:
# Mount Google Drive when running in Colab.
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Google Colab; Drive mount skipped.")

ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")
if not ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found: {ROOT}\n"
        "Confirm that Google Drive is mounted and the project folder name is unchanged."
    )

print(f"Project root: {ROOT}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Frozen upstream identities and Cell 7B4 constants

In [2]:
from __future__ import annotations

import csv
import hashlib
import importlib.metadata as importlib_metadata
import json
import os
import platform
import re
import shutil
import sys
import unicodedata
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import pyarrow.parquet as pq


CELL_ID = "7B4"
PACKAGE_VERSION = "1.0.0"
NOTEBOOK_NAME = "05_GES_Aware_Genomic_RAG_Cell_7B4_Configuration_Freeze.ipynb"
FROZEN_SEED = 20260722

EXPECTED_CELL_7B3_DECISION_PREFIX = (
    "PASS_STAGE7B3_SCORE_BLIND_100920_EVIDENCE_PACKETS_AND_SEMANTIC_CORPUS_"
)

EXPECTED_CELL_7B3 = OrderedDict([
    (
        "evidence_packets",
        {
            "filename": "cell_7b3_score_blind_evidence_packets_v1.parquet",
            "sha256": "f4d5580a5a6ec10f87ff79a1e29fbaab54a5ab2cce3646de018c1685ee60dcb8",
            "expected_rows": 100_920,
            "expected_columns": 29,
        },
    ),
    (
        "semantic_corpus",
        {
            "filename": "cell_7b3_score_blind_semantic_corpus_v1.parquet",
            "sha256": "2fead04f6c0814bb87207c9c36db7475370ae626f6a326c89ce672f402339399",
            "expected_rows": 100_920,
        },
    ),
    (
        "primary_questions",
        {
            "filename": "cell_7b3_primary_question_set_v1.csv",
            "sha256": "c76e81952fcc6a698866b64da7b7daabeb281b7b9d10e17873596095b69d95df",
            "expected_rows": 80,
        },
    ),
    (
        "ordered_reserves",
        {
            "filename": "cell_7b3_ordered_question_reserve_inventory_v1.csv",
            "sha256": "9528908718b834f3aa332a73907d66a413581e4311b0939bd02f9113b5aac6f3",
            "expected_rows": 80,
        },
    ),
    (
        "structured_answer_keys",
        {
            "filename": "cell_7b3_structured_answer_keys_v1.parquet",
            "sha256": "bccf3691336ed8e21a4e7b2876fcf6883ee9a8447938a0edf61d41fc5e0d0332",
            "expected_rows": 80,
        },
    ),
    (
        "rubric_assignments",
        {
            "filename": "cell_7b3_question_rubric_assignment_inventory_v1.csv",
            "sha256": "4185ad173c5e3e2ecea31268b8d833ba111839f5ddd1b96c3fda083e01afcba9",
            "expected_rows": 640,
        },
    ),
    (
        "materialization_report",
        {
            "filename": "cell_7b3_score_blind_materialization_report_v1.json",
            "sha256": "0b9dc618ef4b3dd2dd6b33e01c95696bdee73b0592e39df7f164d3208742c3eb",
        },
    ),
    (
        "qc",
        {
            "filename": "cell_7b3_score_blind_materialization_qc_v1.json",
            "sha256": "7402dc1e4651020b0e939164c5cf38d05d9e0085222345fb44a43ff0a4f829d0",
        },
    ),
    (
        "manifest",
        {
            "filename": "cell_7b3_score_blind_materialization_manifest_v1.json",
            "sha256": "9d0674014f6650cde12f96abb75c06215c36870f7ea2af0dec1916e2b756a2c5",
        },
    ),
])

CONFIG_DIR = (
    ROOT
    / "configs"
    / "stage7_rag"
    / "cell_7b4_configuration_freeze_v1"
)
QC_DIR = (
    ROOT
    / "outputs"
    / "quality_checks"
    / "stage7_rag"
    / "cell_7b4_configuration_freeze_v1"
)

OUTPUTS = OrderedDict([
    (
        "embedding_retrieval",
        CONFIG_DIR / "cell_7b4_embedding_retrieval_configuration_v1.json",
    ),
    (
        "quality_reranking",
        CONFIG_DIR / "cell_7b4_quality_reranking_configuration_v1.json",
    ),
    (
        "llm_prompt_response",
        CONFIG_DIR / "cell_7b4_llm_prompt_response_configuration_v1.json",
    ),
    (
        "runtime_determinism",
        CONFIG_DIR / "cell_7b4_runtime_and_determinism_configuration_v1.json",
    ),
    (
        "condition_aliases",
        CONFIG_DIR / "cell_7b4_condition_alias_inventory_v1.csv",
    ),
    (
        "requirements_lock",
        CONFIG_DIR / "cell_7b4_execution_requirements_lock_v1.txt",
    ),
    (
        "qc",
        QC_DIR / "cell_7b4_configuration_freeze_qc_v1.json",
    ),
    (
        "manifest",
        CONFIG_DIR / "cell_7b4_configuration_freeze_manifest_v1.json",
    ),
])

for directory in (CONFIG_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if OUTPUTS["manifest"].exists():
    raise FileExistsError(
        f"Cell 7B4 manifest already exists: {OUTPUTS['manifest']}\n"
        "Fail-closed overwrite protection is active. Do not overwrite a frozen Cell 7B4 package."
    )

print("Frozen Cell 7B3 identities loaded.")
print(f"Cell 7B4 config directory: {CONFIG_DIR}")
print(f"Cell 7B4 QC directory    : {QC_DIR}")

Frozen Cell 7B3 identities loaded.
Cell 7B4 config directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7b4_configuration_freeze_v1
Cell 7B4 QC directory    : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7b4_configuration_freeze_v1


## 2. Metadata-only upstream reverification

In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + ".sha256")


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        raise ValueError(f"Empty SHA-256 sidecar: {path}")
    token = text.split()[0].strip()
    if not re.fullmatch(r"[0-9a-fA-F]{64}", token):
        raise ValueError(f"Invalid SHA-256 sidecar format: {path}")
    return token.lower()


def sidecar_is_valid(path: Path) -> bool:
    sc = sidecar_path(path)
    return sc.exists() and read_sidecar_hash(sc) == sha256_file(path)


def locate_exact_hash(filename: str, expected_hash: str) -> Path:
    candidates = sorted(ROOT.rglob(filename))
    matches = [p for p in candidates if p.is_file() and sha256_file(p) == expected_hash]
    if len(matches) != 1:
        details = "\n".join(str(p) for p in candidates) or "<none>"
        raise RuntimeError(
            f"Expected exactly one checksum-matching artifact for {filename}; "
            f"found {len(matches)}.\nCandidates:\n{details}"
        )
    return matches[0]


def csv_data_row_count(path: Path) -> int:
    # Structural row count only; no CSV fields are inspected.
    with path.open("rb") as handle:
        line_count = sum(1 for _ in handle)
    return max(0, line_count - 1)


def parquet_metadata(path: Path) -> dict[str, Any]:
    metadata = pq.ParquetFile(path).metadata
    schema_names = list(pq.ParquetFile(path).schema_arrow.names)
    return {
        "rows": int(metadata.num_rows),
        "columns": int(metadata.num_columns),
        "row_groups": int(metadata.num_row_groups),
        "schema_names": schema_names,
    }


upstream_paths: OrderedDict[str, Path] = OrderedDict()
upstream_inventory: list[dict[str, Any]] = []

for key, spec in EXPECTED_CELL_7B3.items():
    path = locate_exact_hash(spec["filename"], spec["sha256"])
    upstream_paths[key] = path

    if not sidecar_is_valid(path):
        raise AssertionError(f"Missing or invalid SHA-256 sidecar: {path}")

    stat = path.stat()
    record: dict[str, Any] = {
        "artifact_id": key,
        "filename": path.name,
        "path": str(path),
        "sha256": sha256_file(path),
        "bytes": int(stat.st_size),
        "modified_time_ns": int(stat.st_mtime_ns),
        "sidecar_path": str(sidecar_path(path)),
        "sidecar_sha256": sha256_file(sidecar_path(path)),
    }

    if path.suffix.lower() == ".parquet":
        meta = parquet_metadata(path)
        record["parquet_rows"] = meta["rows"]
        record["parquet_columns"] = meta["columns"]
        record["parquet_row_groups"] = meta["row_groups"]
        record["parquet_schema_names"] = meta["schema_names"]
    elif path.suffix.lower() == ".csv":
        record["csv_data_rows"] = csv_data_row_count(path)

    upstream_inventory.append(record)

cell_7b3_manifest = json.loads(
    upstream_paths["manifest"].read_text(encoding="utf-8")
)
manifest_text = json.dumps(cell_7b3_manifest, sort_keys=True).lower()
terminal_decision = str(
    cell_7b3_manifest.get(
        "terminal_decision",
        cell_7b3_manifest.get("decision", ""),
    )
)

if not terminal_decision.startswith(EXPECTED_CELL_7B3_DECISION_PREFIX):
    raise AssertionError(
        "Cell 7B3 manifest does not contain the expected frozen PASS decision."
    )
if "7b4" not in manifest_text or "configuration" not in manifest_text:
    raise AssertionError(
        "Cell 7B3 manifest does not preserve the Cell 7B4 configuration-only boundary."
    )

packet_meta = parquet_metadata(upstream_paths["evidence_packets"])
corpus_meta = parquet_metadata(upstream_paths["semantic_corpus"])
answer_key_meta = parquet_metadata(upstream_paths["structured_answer_keys"])

prohibited_schema_tokens = (
    "full_ges",
    "no_star_ges",
    "combined_metadata",
    "instability_risk",
    "p_stable",
    "quality_rank",
    "semantic_rank",
    "future_outcome",
    "recommendation",
)
packet_schema_lower = [name.lower() for name in packet_meta["schema_names"]]
corpus_schema_lower = [name.lower() for name in corpus_meta["schema_names"]]
prohibited_packet_columns = [
    name for name in packet_schema_lower
    if any(token in name for token in prohibited_schema_tokens)
]
prohibited_corpus_columns = [
    name for name in corpus_schema_lower
    if any(token in name for token in prohibited_schema_tokens)
]

structural_checks = OrderedDict([
    ("cell_7b3_manifest_pass", terminal_decision.startswith(EXPECTED_CELL_7B3_DECISION_PREFIX)),
    ("cell_7b3_authorizes_7b4_configuration", "7b4" in manifest_text and "configuration" in manifest_text),
    ("all_nine_artifacts_found", len(upstream_paths) == 9),
    ("all_nine_hashes_exact", all(
        sha256_file(upstream_paths[key]) == spec["sha256"]
        for key, spec in EXPECTED_CELL_7B3.items()
    )),
    ("all_nine_sidecars_valid", all(sidecar_is_valid(path) for path in upstream_paths.values())),
    ("evidence_packet_rows_100920", packet_meta["rows"] == 100_920),
    ("evidence_packet_columns_29", packet_meta["columns"] == 29),
    ("semantic_corpus_rows_100920", corpus_meta["rows"] == 100_920),
    ("primary_question_rows_80", csv_data_row_count(upstream_paths["primary_questions"]) == 80),
    ("reserve_rows_80", csv_data_row_count(upstream_paths["ordered_reserves"]) == 80),
    ("answer_key_rows_80_metadata_only", answer_key_meta["rows"] == 80),
    ("rubric_assignment_rows_640", csv_data_row_count(upstream_paths["rubric_assignments"]) == 640),
    ("packet_schema_score_blind", len(prohibited_packet_columns) == 0),
    ("semantic_corpus_schema_score_blind", len(prohibited_corpus_columns) == 0),
    ("answer_key_content_not_loaded", True),
])

failed_structural = [
    name for name, passed in structural_checks.items() if not bool(passed)
]
if failed_structural:
    raise RuntimeError(
        "Cell 7B4 upstream preflight failed:\n- "
        + "\n- ".join(failed_structural)
    )

immutable_upstream_hashes_before = OrderedDict(
    (key, sha256_file(path)) for key, path in upstream_paths.items()
)

print("Cell 7B3 metadata-only reverification: PASS")
print(f"Artifacts verified      : {len(upstream_paths)}/9")
print(f"Evidence packets        : {packet_meta['rows']:,} rows × {packet_meta['columns']} columns")
print(f"Semantic corpus         : {corpus_meta['rows']:,} rows")
print(f"Primary questions       : {csv_data_row_count(upstream_paths['primary_questions']):,}")
print(f"Ordered reserves        : {csv_data_row_count(upstream_paths['ordered_reserves']):,}")
print(f"Structured answer keys  : {answer_key_meta['rows']:,} rows (metadata only)")
print(f"Rubric assignments      : {csv_data_row_count(upstream_paths['rubric_assignments']):,}")

Cell 7B3 metadata-only reverification: PASS
Artifacts verified      : 9/9
Evidence packets        : 100,920 rows × 29 columns
Semantic corpus         : 100,920 rows
Primary questions       : 80
Ordered reserves        : 80
Structured answer keys  : 80 rows (metadata only)
Rubric assignments      : 640


## 3. Freeze exact embedding, retrieval, and quality-reranking implementation

In [4]:
NORMALIZATION_REFERENCE_IMPLEMENTATION = r"""
def normalize_semantic_text(value: str) -> str:
    if not isinstance(value, str):
        raise TypeError("semantic text must be a Python string")
    value.encode("utf-8", errors="strict")
    text = unicodedata.normalize("NFKC", value)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\s+", " ", text, flags=re.UNICODE).strip()
    return text
""".strip()

RANDOM_QUALITY_REFERENCE_IMPLEMENTATION = r"""
def deterministic_random_quality(question_id: str, rcv_accession: str) -> float:
    payload = f"20260722|{question_id}|{rcv_accession}".encode("utf-8")
    integer = int(hashlib.sha256(payload).hexdigest()[:16], 16)
    return integer / float((1 << 64) - 1)
""".strip()

RRF_REFERENCE_IMPLEMENTATION = r"""
def weighted_rrf(semantic_rank: int, quality_rank: int) -> float:
    return 0.75 / (60 + semantic_rank) + 0.25 / (60 + quality_rank)
""".strip()

embedding_retrieval_config = {
    "cell_id": CELL_ID,
    "package_version": PACKAGE_VERSION,
    "scope": "configuration_freeze_only",
    "embedding": {
        "provider": "huggingface_hub",
        "library": "sentence-transformers",
        "model_id": "NeuML/pubmedbert-base-embeddings",
        "revision": "b79526d6ef3645e0df4530322e266f24c829f5ef",
        "trust_remote_code": False,
        "expected_embedding_dimension": 768,
        "max_sequence_length_tokens": 512,
        "pooling": "model-native mean pooling",
        "do_lower_case": False,
        "input_dtype": "UTF-8 string",
        "output_dtype": "float32",
        "normalize_embeddings_l2": True,
        "batch_size": 64,
        "show_progress_bar": True,
        "device_policy": "cuda_if_available_else_cpu",
        "query_prefix": "",
        "document_prefix": "",
        "model_download_performed_in_cell_7b4": False,
        "embeddings_generated_in_cell_7b4": False,
    },
    "text_normalization": {
        "order": [
            "validate Python string and strict UTF-8 encodability",
            "Unicode NFKC normalization",
            "convert CRLF and CR to LF",
            "collapse every Unicode whitespace run to one ASCII space",
            "strip leading and trailing whitespace",
        ],
        "case_policy": "preserve source case",
        "punctuation_policy": "preserve punctuation",
        "identifier_policy": "preserve HGVS expressions, ClinVar accessions, gene symbols, and condition identifiers",
        "score_injection": "prohibited",
        "question_injection_into_document_text": "prohibited",
        "reference_implementation": NORMALIZATION_REFERENCE_IMPLEMENTATION,
        "reference_implementation_sha256": hashlib.sha256(
            NORMALIZATION_REFERENCE_IMPLEMENTATION.encode("utf-8")
        ).hexdigest(),
    },
    "semantic_similarity": {
        "metric": "cosine_similarity",
        "implementation": "inner product of L2-normalized float32 vectors",
        "faiss_index": "IndexFlatIP",
        "approximate_nearest_neighbor": False,
        "corpus_index_scope": "all 100,920 frozen semantic-corpus rows",
        "query_encoding": "same pinned model, revision, normalization, dtype, and L2 normalization as corpus",
        "descending_sort": True,
        "tie_breaker_order": [
            "rcv_accession ascending",
            "packet_id ascending",
        ],
    },
    "candidate_pool": {
        "semantic_candidate_pool_k": 20,
        "final_context_k": 5,
        "same_top20_pool_for_all_conditions": True,
        "candidate_pool_created_once_per_question": True,
        "candidate_pool_reused_without_mutation": True,
        "hard_evidence_exclusion": False,
        "score_threshold_filtering": False,
        "missing_quality_score_policy": "fail_closed; do not silently impute or drop",
    },
    "forbidden_operations": {
        "load_cell_7a3_score_table": True,
        "generate_embeddings": True,
        "build_faiss_index": True,
        "encode_questions": True,
        "execute_similarity_search": True,
        "inspect_retrieval_results": True,
    },
}

condition_alias_rows = [
    {
        "condition_id": "A",
        "blinded_alias": "ARM-MICA",
        "condition_name": "Semantic-only RAG",
        "role": "baseline",
        "quality_signal": "none",
        "final_order_policy": "semantic rank only",
    },
    {
        "condition_id": "B",
        "blinded_alias": "ARM-ORBIT",
        "condition_name": "Review/conflict-aware RAG",
        "role": "mandatory metadata comparator",
        "quality_signal": "1 - mean(review_stars_instability_risk, conflict_instability_risk)",
        "final_order_policy": "weighted RRF",
    },
    {
        "condition_id": "C",
        "blinded_alias": "ARM-KITE",
        "condition_name": "Combined-metadata-aware RAG",
        "role": "mandatory strongest metadata comparator",
        "quality_signal": "1 - combined_metadata_instability_risk",
        "final_order_policy": "weighted RRF",
    },
    {
        "condition_id": "D",
        "blinded_alias": "ARM-PULSE",
        "condition_name": "Full-GES-aware RAG",
        "role": "primary intervention",
        "quality_signal": "full_ges_p_stable_t1",
        "final_order_policy": "weighted RRF",
    },
    {
        "condition_id": "E",
        "blinded_alias": "ARM-LARCH",
        "condition_name": "No-star-GES-aware RAG",
        "role": "feature-ablation comparator",
        "quality_signal": "no_star_ges_p_stable_t1",
        "final_order_policy": "weighted RRF",
    },
    {
        "condition_id": "F",
        "blinded_alias": "ARM-NOVA",
        "condition_name": "Random-quality control RAG",
        "role": "negative control",
        "quality_signal": "deterministic SHA-256 pseudo-random quality score",
        "final_order_policy": "weighted RRF",
    },
]

quality_reranking_config = {
    "cell_id": CELL_ID,
    "package_version": PACKAGE_VERSION,
    "scope": "configuration_freeze_only",
    "candidate_pool_invariance": {
        "input_pool": "same frozen semantic top-20 for all six conditions",
        "input_order": "semantic rank ascending",
        "final_context_k": 5,
        "hard_exclusion": False,
        "quality_score_exposed_to_llm": False,
    },
    "quality_rank": {
        "ranking_scope": "within each question's fixed top-20 candidate pool",
        "direction": "higher quality signal receives better rank",
        "rank_origin": 1,
        "method": "ordinal rank after deterministic full sort",
        "tie_breaker_order": [
            "rcv_accession ascending",
            "packet_id ascending",
        ],
    },
    "weighted_reciprocal_rank_fusion": {
        "formula": "0.75/(60 + semantic_rank) + 0.25/(60 + quality_rank)",
        "semantic_weight": 0.75,
        "quality_weight": 0.25,
        "rrf_constant": 60,
        "descending_sort": True,
        "final_tie_breaker_order": [
            "semantic_rank ascending",
            "quality_rank ascending",
            "rcv_accession ascending",
            "packet_id ascending",
        ],
        "reference_implementation": RRF_REFERENCE_IMPLEMENTATION,
        "reference_implementation_sha256": hashlib.sha256(
            RRF_REFERENCE_IMPLEMENTATION.encode("utf-8")
        ).hexdigest(),
    },
    "condition_a_semantic_only": {
        "quality_rank_constructed": False,
        "rrf_applied": False,
        "final_context": "semantic ranks 1 through 5",
    },
    "conditions_b_to_f": {
        "rrf_applied": True,
        "final_context": "top five by frozen weighted RRF",
    },
    "random_quality_control": {
        "seed": FROZEN_SEED,
        "hash_algorithm": "SHA-256",
        "payload": "20260722|{question_id}|{rcv_accession}",
        "integer_source": "first 16 hexadecimal characters interpreted as unsigned 64-bit integer",
        "scale": "integer / (2^64 - 1)",
        "reference_implementation": RANDOM_QUALITY_REFERENCE_IMPLEMENTATION,
        "reference_implementation_sha256": hashlib.sha256(
            RANDOM_QUALITY_REFERENCE_IMPLEMENTATION.encode("utf-8")
        ).hexdigest(),
    },
    "condition_inventory": condition_alias_rows,
    "primary_comparison": "D_vs_A",
    "mandatory_secondary_comparisons": [
        "D_vs_B",
        "D_vs_C",
        "D_vs_E",
        "D_vs_F",
    ],
    "forbidden_operations": {
        "load_scores_in_cell_7b4": True,
        "construct_quality_ranks": True,
        "apply_reranking": True,
        "inspect_top20_or_top5_results": True,
        "tune_weights_or_rrf_constant": True,
        "change_top20_or_top5": True,
    },
}

print("Embedding/retrieval and quality-reranking configurations constructed in memory.")
print("Scientific execution performed: NO")

Embedding/retrieval and quality-reranking configurations constructed in memory.
Scientific execution performed: NO


## 4. Freeze prompt templates, strict response schema, LLM snapshot, and generation policy

In [5]:
SYSTEM_PROMPT = """
You are evaluating a research-only genomic evidence-synthesis task using only the evidence records supplied in the prompt.

Requirements:
1. Do not use outside knowledge, web search, tools, or unstated assumptions.
2. Do not provide patient-specific diagnosis, treatment advice, or autonomous clinical recommendations.
3. State the aggregate interpretation supported by the supplied records.
4. Explicitly disclose conflicting evidence when it is present.
5. Distinguish evidence strength from the classification label.
6. Use a cautious answer or abstain when the supplied evidence is insufficient, weakly reviewed, internally conflicting, or unable to support a direct answer.
7. Cite only the supplied EVIDENCE_ID values. Every material factual claim must be supported by at least one cited evidence record.
8. Never infer or mention the experimental condition, quality score, GES score, metadata score, semantic rank, or quality rank.
9. Return only an object conforming to the provided strict JSON schema.
""".strip()

USER_PROMPT_TEMPLATE = """
QUESTION_ID: {question_id}

QUESTION:
{question_text}

FROZEN EVIDENCE CONTEXT:
{context_blocks}

Answer the question using only the frozen evidence context. Follow the system requirements and return strict JSON.
""".strip()

CONTEXT_BLOCK_TEMPLATE = """
[CONTEXT_POSITION={context_position}]
[EVIDENCE_ID={packet_id}]
[RCV_ACCESSION={rcv_accession}]
{semantic_text}
""".strip()

RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "answer": {
            "type": "string",
            "description": "Concise answer supported by the supplied evidence.",
        },
        "clinical_significance": {
            "type": "string",
            "description": "Aggregate clinical-significance interpretation stated by the evidence.",
        },
        "conflict_detected": {
            "type": "boolean",
            "description": "Whether the supplied evidence contains disagreement or conflict.",
        },
        "evidence_strength": {
            "type": "string",
            "enum": ["high", "moderate", "low"],
        },
        "response_policy": {
            "type": "string",
            "enum": ["answer", "cautious_answer", "abstain"],
        },
        "confidence": {
            "type": "number",
            "minimum": 0.0,
            "maximum": 1.0,
            "description": "Secondary self-reported confidence; not treated as a calibrated probability.",
        },
        "evidence_ids": {
            "type": "array",
            "items": {"type": "string"},
            "uniqueItems": True,
        },
        "reasoning_summary": {
            "type": "string",
            "description": "Brief evidence-grounded explanation without hidden chain-of-thought.",
        },
    },
    "required": [
        "answer",
        "clinical_significance",
        "conflict_detected",
        "evidence_strength",
        "response_policy",
        "confidence",
        "evidence_ids",
        "reasoning_summary",
    ],
    "additionalProperties": False,
}

llm_prompt_response_config = {
    "cell_id": CELL_ID,
    "package_version": PACKAGE_VERSION,
    "scope": "configuration_freeze_only",
    "llm": {
        "provider": "OpenAI",
        "api": "Responses API",
        "model": "gpt-4.1-mini-2025-04-14",
        "model_aliases_prohibited": [
            "gpt-4.1-mini",
            "latest",
        ],
        "fixed_snapshot_required": True,
        "tools": [],
        "web_search": False,
        "file_search": False,
        "code_interpreter": False,
        "store": False,
        "stream": False,
    },
    "generation": {
        "temperature": 0.0,
        "top_p": 1.0,
        "max_output_tokens": 1200,
        "presence_penalty": 0.0,
        "frequency_penalty": 0.0,
        "repetitions_per_question_condition": 3,
        "run_ids": [0, 1, 2],
        "timeout_seconds": 120,
        "maximum_transport_retries": 3,
        "retry_backoff_seconds": [2, 5, 10],
        "retry_policy": "retry transport/rate-limit failures only; never retry to obtain a preferred answer",
        "request_idempotency_key": "SHA256(cell_id|question_id|blinded_alias|run_id|prompt_sha256)",
    },
    "run_aggregation": {
        "primary_binary_endpoint": "majority of three independently scored runs",
        "continuous_metrics": "arithmetic mean across the three runs",
        "categorical_output_audit": "modal value; ties resolved by lowest run_id",
        "run_to_run_disagreement": "reported as a robustness outcome",
        "individual_outputs_retained": True,
    },
    "prompt": {
        "system_prompt": SYSTEM_PROMPT,
        "system_prompt_sha256": hashlib.sha256(
            SYSTEM_PROMPT.encode("utf-8")
        ).hexdigest(),
        "user_prompt_template": USER_PROMPT_TEMPLATE,
        "user_prompt_template_sha256": hashlib.sha256(
            USER_PROMPT_TEMPLATE.encode("utf-8")
        ).hexdigest(),
        "context_block_template": CONTEXT_BLOCK_TEMPLATE,
        "context_block_template_sha256": hashlib.sha256(
            CONTEXT_BLOCK_TEMPLATE.encode("utf-8")
        ).hexdigest(),
        "question_position": "before evidence context",
        "context_order": "frozen final top-5 order",
        "score_values_in_prompt": False,
        "condition_identity_in_prompt": False,
        "answer_key_in_prompt": False,
    },
    "structured_output": {
        "type": "json_schema",
        "name": "ges_rag_genomic_evidence_answer",
        "strict": True,
        "schema": RESPONSE_SCHEMA,
        "schema_sha256": hashlib.sha256(
            json.dumps(
                RESPONSE_SCHEMA,
                sort_keys=True,
                separators=(",", ":"),
            ).encode("utf-8")
        ).hexdigest(),
    },
    "forbidden_operations": {
        "materialize_question_specific_prompts": True,
        "load_or_inspect_answer_key_outcomes": True,
        "call_openai_api": True,
        "generate_responses": True,
        "modify_prompts_after_test_output_review": True,
    },
}

print("Prompt, response schema, fixed LLM snapshot, and generation policy constructed in memory.")
print("Prompt materialization performed: NO")
print("LLM called: NO")

Prompt, response schema, fixed LLM snapshot, and generation policy constructed in memory.
Prompt materialization performed: NO
LLM called: NO


## 5. Freeze runtime targets and deterministic controls

In [6]:
PACKAGE_TARGETS = OrderedDict([
    ("sentence-transformers", "5.6.0"),
    ("transformers", "5.14.1"),
    ("faiss-cpu", "1.14.3"),
    ("openai", "2.45.0"),
])

OBSERVED_PACKAGES = [
    "numpy",
    "pandas",
    "pyarrow",
    "torch",
    "sentence-transformers",
    "transformers",
    "huggingface-hub",
    "faiss-cpu",
    "openai",
    "pydantic",
]

observed_runtime_packages = OrderedDict()
for package_name in OBSERVED_PACKAGES:
    try:
        observed_runtime_packages[package_name] = importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        observed_runtime_packages[package_name] = None

requirements_lines = [
    "# Cell 7B4 frozen execution requirements",
    "# Install these exact future-execution packages before any Cell 7C execution.",
]
for package_name, version in PACKAGE_TARGETS.items():
    requirements_lines.append(f"{package_name}=={version}")
REQUIREMENTS_LOCK_TEXT = "\n".join(requirements_lines) + "\n"

runtime_determinism_config = {
    "cell_id": CELL_ID,
    "package_version": PACKAGE_VERSION,
    "scope": "configuration_freeze_only",
    "runtime_observed_during_configuration_freeze": {
        "python_version": platform.python_version(),
        "python_implementation": platform.python_implementation(),
        "platform": platform.platform(),
        "machine": platform.machine(),
        "processor": platform.processor(),
        "executable": sys.executable,
        "packages": observed_runtime_packages,
    },
    "future_execution_package_targets": PACKAGE_TARGETS,
    "requirements_lock_sha256": hashlib.sha256(
        REQUIREMENTS_LOCK_TEXT.encode("utf-8")
    ).hexdigest(),
    "deterministic_controls": {
        "global_seed": FROZEN_SEED,
        "python_hash_seed": "0",
        "numpy_seed": FROZEN_SEED,
        "torch_manual_seed": FROZEN_SEED,
        "torch_cuda_manual_seed_all": FROZEN_SEED,
        "torch_use_deterministic_algorithms": True,
        "torch_cudnn_deterministic": True,
        "torch_cudnn_benchmark": False,
        "faiss_index_type": "IndexFlatIP exact search",
        "corpus_sort_before_embedding": [
            "packet_id ascending",
            "rcv_accession ascending",
        ],
        "question_sort_before_execution": [
            "question_id ascending",
        ],
        "condition_execution_order": [
            "ARM-MICA",
            "ARM-ORBIT",
            "ARM-KITE",
            "ARM-PULSE",
            "ARM-LARCH",
            "ARM-NOVA",
        ],
        "run_order": [0, 1, 2],
        "stable_json": "UTF-8, sort_keys=True, indent=2, trailing newline",
        "stable_csv": "UTF-8, newline='', fixed header order, lineterminator='\\n'",
    },
    "leakage_and_invariance_controls": {
        "cell_7a3_scores_separate_from_score_blind_package": True,
        "same_semantic_top20_across_conditions": True,
        "scores_not_exposed_to_llm": True,
        "answer_keys_unavailable_to_retrieval_and_generation_code": True,
        "no_hard_evidence_exclusion": True,
        "no_threshold_tuning": True,
        "no_weight_tuning": True,
        "no_prompt_tuning_after_locked_output_review": True,
        "no_question_replacement": True,
        "no_metric_changes": True,
        "egfr_reported_separately_as_exploratory": True,
        "research_only_no_patient_level_decision_support": True,
    },
    "cell_7b4_scientific_operations": {
        "cell_7a3_scores_loaded": False,
        "answer_key_outcomes_inspected": False,
        "embedding_model_downloaded": False,
        "embeddings_generated": False,
        "faiss_index_constructed": False,
        "semantic_retrieval_executed": False,
        "quality_reranking_applied": False,
        "prompts_materialized": False,
        "llm_called": False,
        "adjudication_performed": False,
        "rag_metrics_calculated": False,
    },
    "next_stage": {
        "authorized_by_cell_7b4": False,
        "required_action": (
            "Create a separate fail-closed execution-authorization cell that reverifies "
            "the frozen Cell 7B4 manifest before any embedding, retrieval, reranking, "
            "prompt materialization, or LLM generation."
        ),
    },
}

print("Runtime targets and deterministic controls constructed in memory.")

Runtime targets and deterministic controls constructed in memory.


## 6. QC, checksum-protected materialization, and final freeze

In [7]:
def atomic_write_bytes(path: Path, payload: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as handle:
        handle.write(payload)
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)


def stable_write_json(path: Path, payload: Any) -> str:
    data = (
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )
        + "\n"
    ).encode("utf-8")
    atomic_write_bytes(path, data)
    return sha256_file(path)


def stable_write_text(path: Path, text: str) -> str:
    atomic_write_bytes(path, text.encode("utf-8"))
    return sha256_file(path)


def stable_write_csv(path: Path, rows: list[dict[str, Any]], fieldnames: list[str]) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=fieldnames,
            extrasaction="raise",
            lineterminator="\n",
        )
        writer.writeheader()
        writer.writerows(rows)
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)
    return sha256_file(path)


def write_sidecar(path: Path) -> str:
    digest = sha256_file(path)
    stable_write_text(sidecar_path(path), f"{digest}  {path.name}\n")
    return sha256_file(sidecar_path(path))


aliases = [row["blinded_alias"] for row in condition_alias_rows]
condition_ids = [row["condition_id"] for row in condition_alias_rows]

prewrite_checks = OrderedDict()
prewrite_checks.update(structural_checks)
prewrite_checks.update(OrderedDict([
    ("embedding_model_pinned", embedding_retrieval_config["embedding"]["model_id"] == "NeuML/pubmedbert-base-embeddings"),
    ("embedding_revision_full_commit", re.fullmatch(r"[0-9a-f]{40}", embedding_retrieval_config["embedding"]["revision"]) is not None),
    ("embedding_dimension_768", embedding_retrieval_config["embedding"]["expected_embedding_dimension"] == 768),
    ("embedding_max_length_512", embedding_retrieval_config["embedding"]["max_sequence_length_tokens"] == 512),
    ("embedding_l2_normalized", embedding_retrieval_config["embedding"]["normalize_embeddings_l2"] is True),
    ("exact_index_flat_ip", embedding_retrieval_config["semantic_similarity"]["faiss_index"] == "IndexFlatIP"),
    ("approximate_search_disabled", embedding_retrieval_config["semantic_similarity"]["approximate_nearest_neighbor"] is False),
    ("candidate_pool_top20", embedding_retrieval_config["candidate_pool"]["semantic_candidate_pool_k"] == 20),
    ("final_context_top5", embedding_retrieval_config["candidate_pool"]["final_context_k"] == 5),
    ("common_pool_invariant", embedding_retrieval_config["candidate_pool"]["same_top20_pool_for_all_conditions"] is True),
    ("hard_exclusion_prohibited", embedding_retrieval_config["candidate_pool"]["hard_evidence_exclusion"] is False),
    ("six_conditions", len(condition_alias_rows) == 6),
    ("condition_ids_exact", condition_ids == ["A", "B", "C", "D", "E", "F"]),
    ("six_unique_blinded_aliases", len(set(aliases)) == 6),
    ("primary_intervention_alias_present", any(row["condition_id"] == "D" and row["blinded_alias"] == "ARM-PULSE" for row in condition_alias_rows)),
    ("rrf_semantic_weight_075", quality_reranking_config["weighted_reciprocal_rank_fusion"]["semantic_weight"] == 0.75),
    ("rrf_quality_weight_025", quality_reranking_config["weighted_reciprocal_rank_fusion"]["quality_weight"] == 0.25),
    ("rrf_weights_sum_one", abs(
        quality_reranking_config["weighted_reciprocal_rank_fusion"]["semantic_weight"]
        + quality_reranking_config["weighted_reciprocal_rank_fusion"]["quality_weight"]
        - 1.0
    ) < 1e-12),
    ("rrf_constant_60", quality_reranking_config["weighted_reciprocal_rank_fusion"]["rrf_constant"] == 60),
    ("primary_comparison_d_vs_a", quality_reranking_config["primary_comparison"] == "D_vs_A"),
    ("random_control_seed_frozen", quality_reranking_config["random_quality_control"]["seed"] == FROZEN_SEED),
    ("score_not_exposed_to_llm", quality_reranking_config["candidate_pool_invariance"]["quality_score_exposed_to_llm"] is False),
    ("llm_snapshot_exact", llm_prompt_response_config["llm"]["model"] == "gpt-4.1-mini-2025-04-14"),
    ("llm_aliases_prohibited", "gpt-4.1-mini" in llm_prompt_response_config["llm"]["model_aliases_prohibited"]),
    ("temperature_zero", llm_prompt_response_config["generation"]["temperature"] == 0.0),
    ("three_repetitions", llm_prompt_response_config["generation"]["repetitions_per_question_condition"] == 3),
    ("tools_disabled", llm_prompt_response_config["llm"]["tools"] == []),
    ("web_search_disabled", llm_prompt_response_config["llm"]["web_search"] is False),
    ("strict_json_schema", llm_prompt_response_config["structured_output"]["strict"] is True),
    ("schema_no_additional_properties", RESPONSE_SCHEMA["additionalProperties"] is False),
    ("schema_all_fields_required", set(RESPONSE_SCHEMA["required"]) == set(RESPONSE_SCHEMA["properties"].keys())),
    ("prompt_contains_question_placeholder", "{question_text}" in USER_PROMPT_TEMPLATE),
    ("prompt_contains_context_placeholder", "{context_blocks}" in USER_PROMPT_TEMPLATE),
    ("prompt_has_no_score_placeholder", all(token not in USER_PROMPT_TEMPLATE.lower() for token in ["ges_score", "p_stable", "quality_rank", "semantic_rank"])),
    ("answer_key_not_in_prompt", llm_prompt_response_config["prompt"]["answer_key_in_prompt"] is False),
    ("requirements_four_exact_targets", len(PACKAGE_TARGETS) == 4),
    ("future_execution_not_authorized", runtime_determinism_config["next_stage"]["authorized_by_cell_7b4"] is False),
    ("no_scores_loaded_in_cell_7b4", runtime_determinism_config["cell_7b4_scientific_operations"]["cell_7a3_scores_loaded"] is False),
    ("no_answer_outcomes_inspected", runtime_determinism_config["cell_7b4_scientific_operations"]["answer_key_outcomes_inspected"] is False),
    ("no_embeddings_generated", runtime_determinism_config["cell_7b4_scientific_operations"]["embeddings_generated"] is False),
    ("no_retrieval_executed", runtime_determinism_config["cell_7b4_scientific_operations"]["semantic_retrieval_executed"] is False),
    ("no_reranking_applied", runtime_determinism_config["cell_7b4_scientific_operations"]["quality_reranking_applied"] is False),
    ("no_prompts_materialized", runtime_determinism_config["cell_7b4_scientific_operations"]["prompts_materialized"] is False),
    ("no_llm_called", runtime_determinism_config["cell_7b4_scientific_operations"]["llm_called"] is False),
    ("no_metrics_calculated", runtime_determinism_config["cell_7b4_scientific_operations"]["rag_metrics_calculated"] is False),
]))

failed_prewrite = [
    name for name, passed in prewrite_checks.items() if not bool(passed)
]
if failed_prewrite:
    raise RuntimeError(
        "Cell 7B4 prewrite QC failed:\n- "
        + "\n- ".join(failed_prewrite)
    )

created_utc = datetime.now(timezone.utc).isoformat()

stable_write_json(OUTPUTS["embedding_retrieval"], embedding_retrieval_config)
write_sidecar(OUTPUTS["embedding_retrieval"])

stable_write_json(OUTPUTS["quality_reranking"], quality_reranking_config)
write_sidecar(OUTPUTS["quality_reranking"])

stable_write_json(OUTPUTS["llm_prompt_response"], llm_prompt_response_config)
write_sidecar(OUTPUTS["llm_prompt_response"])

stable_write_json(OUTPUTS["runtime_determinism"], runtime_determinism_config)
write_sidecar(OUTPUTS["runtime_determinism"])

stable_write_csv(
    OUTPUTS["condition_aliases"],
    condition_alias_rows,
    [
        "condition_id",
        "blinded_alias",
        "condition_name",
        "role",
        "quality_signal",
        "final_order_policy",
    ],
)
write_sidecar(OUTPUTS["condition_aliases"])

stable_write_text(OUTPUTS["requirements_lock"], REQUIREMENTS_LOCK_TEXT)
write_sidecar(OUTPUTS["requirements_lock"])

qc_payload = {
    "cell_id": CELL_ID,
    "package_version": PACKAGE_VERSION,
    "created_utc": created_utc,
    "scope": "configuration_freeze_only",
    "checks": [
        {"check": name, "passed": bool(passed)}
        for name, passed in prewrite_checks.items()
    ],
    "passed_checks": int(sum(bool(v) for v in prewrite_checks.values())),
    "failed_checks": int(sum(not bool(v) for v in prewrite_checks.values())),
    "total_checks": int(len(prewrite_checks)),
    "upstream_inventory": upstream_inventory,
    "prohibited_packet_columns": prohibited_packet_columns,
    "prohibited_corpus_columns": prohibited_corpus_columns,
    "scientific_operations": runtime_determinism_config["cell_7b4_scientific_operations"],
    "decision": (
        "PASS_STAGE7B4_EXACT_EMBEDDING_MODEL_REVISION_TEXT_NORMALIZATION_"
        "SIMILARITY_TOP20_CANDIDATE_POOL_TOP5_CONTEXT_QUALITY_RERANKING_"
        "BLINDED_ALIASES_PROMPTS_STRICT_RESPONSE_SCHEMA_FIXED_LLM_SNAPSHOT_"
        "GENERATION_RUNTIME_AND_DETERMINISTIC_CONTROLS_FROZEN_CHECKSUM_PROTECTED_"
        "NO_EMBEDDINGS_RETRIEVAL_RERANKING_PROMPT_MATERIALIZATION_LLM_"
        "ANSWER_KEY_OUTCOME_INSPECTION_OR_RAG_EVALUATION_EXECUTION_NOT_AUTHORIZED"
    ),
}
stable_write_json(OUTPUTS["qc"], qc_payload)
write_sidecar(OUTPUTS["qc"])

output_records = OrderedDict()
for key, path in OUTPUTS.items():
    if key == "manifest":
        continue
    if not sidecar_is_valid(path):
        raise AssertionError(f"Output sidecar verification failed: {path}")
    output_records[key] = {
        "path": str(path),
        "sha256": sha256_file(path),
        "bytes": int(path.stat().st_size),
        "sidecar_path": str(sidecar_path(path)),
        "sidecar_sha256": sha256_file(sidecar_path(path)),
    }

manifest_payload = {
    "cell_id": CELL_ID,
    "package_version": PACKAGE_VERSION,
    "notebook_name": NOTEBOOK_NAME,
    "created_utc": created_utc,
    "purpose": (
        "Freeze exact Experiment 2 embedding, retrieval, quality-reranking, "
        "condition-alias, prompt, response-schema, LLM, generation, runtime, "
        "and deterministic configurations without scientific execution."
    ),
    "upstream_cell_7b3": {
        "manifest_path": str(upstream_paths["manifest"]),
        "manifest_sha256": sha256_file(upstream_paths["manifest"]),
        "terminal_decision": terminal_decision,
        "artifacts": upstream_inventory,
    },
    "output_artifacts": output_records,
    "frozen_design": {
        "conditions": 6,
        "semantic_candidate_pool_k": 20,
        "final_context_k": 5,
        "rrf_formula": "0.75/(60 + semantic_rank) + 0.25/(60 + quality_rank)",
        "embedding_model": "NeuML/pubmedbert-base-embeddings",
        "embedding_revision": "b79526d6ef3645e0df4530322e266f24c829f5ef",
        "llm_model": "gpt-4.1-mini-2025-04-14",
        "repetitions_per_question_condition": 3,
        "primary_comparison": "D_vs_A",
        "hard_exclusion": False,
        "scores_exposed_to_llm": False,
    },
    "qc": {
        "path": str(OUTPUTS["qc"]),
        "sha256": sha256_file(OUTPUTS["qc"]),
        "passed_checks": qc_payload["passed_checks"],
        "failed_checks": qc_payload["failed_checks"],
        "total_checks": qc_payload["total_checks"],
    },
    "scientific_boundary": runtime_determinism_config["cell_7b4_scientific_operations"],
    "terminal_decision": qc_payload["decision"],
    "next_authorized_cell": None,
    "next_required_action": runtime_determinism_config["next_stage"]["required_action"],
}
stable_write_json(OUTPUTS["manifest"], manifest_payload)
write_sidecar(OUTPUTS["manifest"])

readback_embedding = json.loads(
    OUTPUTS["embedding_retrieval"].read_text(encoding="utf-8")
)
readback_reranking = json.loads(
    OUTPUTS["quality_reranking"].read_text(encoding="utf-8")
)
readback_llm = json.loads(
    OUTPUTS["llm_prompt_response"].read_text(encoding="utf-8")
)
readback_runtime = json.loads(
    OUTPUTS["runtime_determinism"].read_text(encoding="utf-8")
)
readback_qc = json.loads(OUTPUTS["qc"].read_text(encoding="utf-8"))
readback_manifest = json.loads(
    OUTPUTS["manifest"].read_text(encoding="utf-8")
)

readback_checks = OrderedDict([
    ("embedding_config_sidecar_valid", sidecar_is_valid(OUTPUTS["embedding_retrieval"])),
    ("reranking_config_sidecar_valid", sidecar_is_valid(OUTPUTS["quality_reranking"])),
    ("llm_config_sidecar_valid", sidecar_is_valid(OUTPUTS["llm_prompt_response"])),
    ("runtime_config_sidecar_valid", sidecar_is_valid(OUTPUTS["runtime_determinism"])),
    ("alias_inventory_sidecar_valid", sidecar_is_valid(OUTPUTS["condition_aliases"])),
    ("requirements_sidecar_valid", sidecar_is_valid(OUTPUTS["requirements_lock"])),
    ("qc_sidecar_valid", sidecar_is_valid(OUTPUTS["qc"])),
    ("manifest_sidecar_valid", sidecar_is_valid(OUTPUTS["manifest"])),
    ("embedding_revision_readback", readback_embedding["embedding"]["revision"] == "b79526d6ef3645e0df4530322e266f24c829f5ef"),
    ("top20_top5_readback", readback_embedding["candidate_pool"]["semantic_candidate_pool_k"] == 20 and readback_embedding["candidate_pool"]["final_context_k"] == 5),
    ("rrf_formula_readback", readback_reranking["weighted_reciprocal_rank_fusion"]["formula"] == "0.75/(60 + semantic_rank) + 0.25/(60 + quality_rank)"),
    ("llm_snapshot_readback", readback_llm["llm"]["model"] == "gpt-4.1-mini-2025-04-14"),
    ("strict_schema_readback", readback_llm["structured_output"]["strict"] is True),
    ("execution_not_authorized_readback", readback_manifest["next_authorized_cell"] is None),
    ("qc_all_passed_readback", readback_qc["failed_checks"] == 0),
    ("manifest_decision_readback", readback_manifest["terminal_decision"] == qc_payload["decision"]),
    ("scientific_operations_all_false", all(
        value is False
        for value in readback_runtime["cell_7b4_scientific_operations"].values()
    )),
])

failed_readback = [
    name for name, passed in readback_checks.items() if not bool(passed)
]
if failed_readback:
    raise RuntimeError(
        "Cell 7B4 readback verification failed:\n- "
        + "\n- ".join(failed_readback)
    )

immutable_upstream_hashes_after = OrderedDict(
    (key, sha256_file(path)) for key, path in upstream_paths.items()
)
if immutable_upstream_hashes_after != immutable_upstream_hashes_before:
    raise AssertionError(
        "One or more frozen Cell 7B3 artifacts changed during Cell 7B4."
    )

total_checks = len(prewrite_checks) + len(readback_checks)
passed_checks = total_checks

separator = "=" * 148
print("\n" + separator)
print("EXPERIMENT 2 — STAGE 7B — CELL 7B4")
print("EMBEDDING, RETRIEVAL, RERANKING, PROMPT, LLM, RUNTIME, AND DETERMINISM CONFIGURATION FREEZE")
print(separator)
print(f"Notebook                                      : {NOTEBOOK_NAME}")
print(f"Project root                                  : {ROOT}")

print("\nUPSTREAM CELL 7B3 REVERIFICATION")
print(f"Cell 7B3 manifest SHA-256                     : {sha256_file(upstream_paths['manifest'])}")
print("Cell 7B3 terminal PASS verified               : YES")
print(f"Frozen Cell 7B3 artifacts                     : {len(upstream_paths)}/9 exact hashes + sidecars")
print(f"Evidence packets                              : {packet_meta['rows']:,} rows × {packet_meta['columns']} columns")
print(f"Semantic corpus                               : {corpus_meta['rows']:,} rows")
print("Answer-key outcomes inspected                 : NO")

print("\nFROZEN CELL 7B4 CONFIGURATION")
print("Embedding model                               : NeuML/pubmedbert-base-embeddings")
print("Embedding revision                            : b79526d6ef3645e0df4530322e266f24c829f5ef")
print("Vector normalization                          : L2")
print("Similarity/index                              : cosine via exact FAISS IndexFlatIP")
print("Semantic candidate pool                       : top-20, identical across six conditions")
print("Final context                                 : top-5")
print("Soft rank fusion                              : 0.75 semantic + 0.25 quality RRF; constant 60")
print("Hard evidence exclusion                       : PROHIBITED")
print("Scores exposed to LLM                         : NO")
print("LLM snapshot                                  : gpt-4.1-mini-2025-04-14")
print("Structured response                           : strict JSON schema")
print("Temperature                                   : 0")
print("Repeated runs                                 : 3 per question-condition")
print("Condition aliases                             : 6 frozen blinded aliases")

print("\nCELL 7B4 FROZEN OUTPUTS")
for label, path in OUTPUTS.items():
    print(f"{label:<46}: {path}")
    print(f"{'SHA-256':<46}: {sha256_file(path)}")

print(f"\nQC checks                                      : {passed_checks}/{total_checks} PASS")

print("\nSCIENTIFIC OPERATIONS")
print("Cell 7A3 scores loaded                        : NO")
print("Embeddings generated                          : NO")
print("FAISS index constructed                       : NO")
print("Semantic retrieval executed                   : NO")
print("Quality reranking applied                     : NO")
print("Prompts materialized                          : NO")
print("LLM called                                     : NO")
print("Adjudication or RAG metrics                    : NO")

print("\nNEXT AUTHORIZATION BOUNDARY")
print("Execution cell                                : NOT AUTHORIZED")
print("Required next action                          : separate fail-closed execution authorization")
print(f"\nFINAL DECISION                                : {qc_payload['decision']}")
print(separator)


EXPERIMENT 2 — STAGE 7B — CELL 7B4
EMBEDDING, RETRIEVAL, RERANKING, PROMPT, LLM, RUNTIME, AND DETERMINISM CONFIGURATION FREEZE
Notebook                                      : 05_GES_Aware_Genomic_RAG_Cell_7B4_Configuration_Freeze.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study

UPSTREAM CELL 7B3 REVERIFICATION
Cell 7B3 manifest SHA-256                     : 9d0674014f6650cde12f96abb75c06215c36870f7ea2af0dec1916e2b756a2c5
Cell 7B3 terminal PASS verified               : YES
Frozen Cell 7B3 artifacts                     : 9/9 exact hashes + sidecars
Evidence packets                              : 100,920 rows × 29 columns
Semantic corpus                               : 100,920 rows
Answer-key outcomes inspected                 : NO

FROZEN CELL 7B4 CONFIGURATION
Embedding model                               : NeuML/pubmedbert-base-embeddings
Embedding revision                            : b79526d6ef3645e0df4530322e266f24c829f5ef
Vector 